In [1]:
import torch 
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [2]:
transform = transforms.ToTensor()

In [3]:
def get_data():
    train_data = datasets.MNIST(
        root='data',
        train=True,
        download=True,
        transform =transform
    )

    test_data = datasets.MNIST(
        root='data',
        train=False,
        download=True,
        transform=transform
    )

    train_loader = DataLoader(
        train_data,
        batch_size=64,
        shuffle=True
    )

    test_loader = DataLoader(
        test_data,
        batch_size=64,
        shuffle=True
    )

    return train_loader,test_loader

In [13]:
class DigitClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.network = nn.Sequential(
            nn.Linear(784,128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )
    def forward(self, x):
        x = self.flatten(x)
        return self.network(x)

In [14]:
model = DigitClassifier()

In [20]:
loss_function = nn.CrossEntropyLoss()

In [17]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [18]:
train_loader, test_loader = get_data()

100%|██████████| 9.91M/9.91M [00:03<00:00, 2.82MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 93.6kB/s]
100%|██████████| 1.65M/1.65M [00:02<00:00, 690kB/s] 
100%|██████████| 4.54k/4.54k [00:00<00:00, 4.51MB/s]


In [22]:
epochs = 5
for epoch in range(epochs):

    model.train()

    for images, labels in train_loader:
        outputs = model(images)
        loss = loss_function(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch : {epoch + 1}, loss : {loss.item():.4f}")

Epoch : 1, loss : 0.0009
Epoch : 2, loss : 0.0086
Epoch : 3, loss : 0.0578
Epoch : 4, loss : 0.0434
Epoch : 5, loss : 0.0193


In [23]:
model.eval()

DigitClassifier(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (network): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [28]:
correct = 0
total = 0
with torch.no_grad():
    for images,labels in test_loader:
        output = model(images)
        predictions = output.argmax(dim = 1)
        total += labels.size(0)
        correct += (predictions == labels).sum().item()

accuracy = correct/total
print(f"Test Accuracy : {accuracy * 100:.2f}%") 

Test Accuracy : 97.36%


In [29]:
torch.save(model.state_dict(), "digit_classifier.pth")

In [30]:
model = DigitClassifier()

In [31]:
model.load_state_dict(
    torch.load("digit_classifier.pth")
)

<All keys matched successfully>

In [32]:
model.eval()

DigitClassifier(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (network): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [63]:
import gradio as gr
import numpy as np
from PIL import Image

In [67]:
def predict_digit(image):
    try:
        image = Image.fromarray(image)
        image = image.resize((28,28))
        image = torch.tensor(__import__("numpy").array(image)).float() / 255.0
        if image.ndim == 2:
            image = image.unsqueeze(0).unsqueeze(0)
        else:
            image = image.permute(2, 0, 1).unsqueeze(0)

        with torch.no_grad():
            output = model(image)

        probabilities = torch.softmax(output, dim = 1)
        prediction = probabilities.argmax(dim=1).item()
        confidence = probabilities[0][prediction].item()
        return prediction, confidence
    except Exception as e:
        print(f"An Error has occured : {e}")

In [68]:
demo = gr.Interface(
    fn=predict_digit,
    inputs=gr.Image(image_mode='L',type="numpy"),
    outputs=[
        gr.Number(label="Prediction"),
        gr.Number(label="Confidence")
    ],
    title="Handwritten Digit Classifier",
    description = "Draw a digit from 0-9"
)

In [69]:
demo.launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.
